In [0]:
import pandas as pd
from sqlalchemy import create_engine, text

RDS_HOST = dbutils.secrets.get(scope="aws-credentials", key="rds-host")
RDS_USER = dbutils.secrets.get(scope="aws-credentials", key="rds-user")
RDS_PASS = dbutils.secrets.get(scope="aws-credentials", key="rds-password")

engine = create_engine(
    f"postgresql+psycopg2://{RDS_USER}:{RDS_PASS}@{RDS_HOST}:5432/enem_db"
)
print("aqui foi 3")

In [0]:
query_uf = """
    SELECT
        sg_uf_residencia                        AS uf,
        nu_ano,
        COUNT(*)                                AS total_candidatos,
        COUNT(*) FILTER (WHERE in_treineiro IS FALSE) AS candidatos_regulares,
        ROUND(AVG(nu_nota_cn)::numeric, 2)      AS media_cn,
        ROUND(AVG(nu_nota_ch)::numeric, 2)      AS media_ch,
        ROUND(AVG(nu_nota_lc)::numeric, 2)      AS media_lc,
        ROUND(AVG(nu_nota_mt)::numeric, 2)      AS media_mt,
        ROUND(AVG(nu_nota_redacao)::numeric, 2) AS media_redacao,
        ROUND(AVG(nu_media_total)::numeric, 2)  AS media_total,
        ROUND(STDDEV(nu_media_total)::numeric, 2) AS desvpad_total
    FROM tru.enem_candidatos
    WHERE sg_uf_residencia IS NOT NULL
    GROUP BY sg_uf_residencia, nu_ano
    ORDER BY media_total DESC
"""
df_uf = pd.read_sql(query_uf, engine)
df_uf.to_sql("enem_por_uf", engine, schema="ref",
             if_exists="replace", index=False)
print(f"ref.enem_por_uf — {len(df_uf)} linhas")

In [0]:
query_mun = """
    SELECT
        co_municipio_residencia                 AS cod_municipio,
        no_municipio_residencia                 AS municipio,
        sg_uf_residencia                        AS uf,
        nu_ano,
        COUNT(*)                                AS total_candidatos,
        ROUND(AVG(nu_nota_cn)::numeric, 2)      AS media_cn,
        ROUND(AVG(nu_nota_ch)::numeric, 2)      AS media_ch,
        ROUND(AVG(nu_nota_lc)::numeric, 2)      AS media_lc,
        ROUND(AVG(nu_nota_mt)::numeric, 2)      AS media_mt,
        ROUND(AVG(nu_nota_redacao)::numeric, 2) AS media_redacao,
        ROUND(AVG(nu_media_total)::numeric, 2)  AS media_total
    FROM tru.enem_candidatos
    WHERE co_municipio_residencia IS NOT NULL
    GROUP BY co_municipio_residencia, no_municipio_residencia,
             sg_uf_residencia, nu_ano
    ORDER BY media_total DESC
"""
df_mun = pd.read_sql(query_mun, engine)
df_mun.to_sql("enem_por_municipio", engine, schema="ref",
              if_exists="replace", index=False)
print(f"ref.enem_por_municipio — {len(df_mun)} linhas")

In [0]:
query_escola = """
    SELECT
        ds_escola,
        sg_uf_residencia                        AS uf,
        nu_ano,
        COUNT(*)                                AS total_candidatos,
        ROUND(AVG(nu_nota_cn)::numeric, 2)      AS media_cn,
        ROUND(AVG(nu_nota_ch)::numeric, 2)      AS media_ch,
        ROUND(AVG(nu_nota_lc)::numeric, 2)      AS media_lc,
        ROUND(AVG(nu_nota_mt)::numeric, 2)      AS media_mt,
        ROUND(AVG(nu_nota_redacao)::numeric, 2) AS media_redacao,
        ROUND(AVG(nu_media_total)::numeric, 2)  AS media_total
    FROM tru.enem_candidatos
    WHERE ds_escola IS NOT NULL
    GROUP BY ds_escola, sg_uf_residencia, nu_ano
    ORDER BY media_total DESC
"""
df_escola = pd.read_sql(query_escola, engine)
df_escola.to_sql("enem_por_escola", engine, schema="ref",
                 if_exists="replace", index=False)
print(f"ref.enem_por_escola — {len(df_escola)} linhas")

In [0]:
query_renda = """
    SELECT
        q006                                    AS faixa_renda,
        nu_ano,
        COUNT(*)                                AS total_candidatos,
        ROUND(AVG(nu_media_total)::numeric, 2)  AS media_total,
        ROUND(AVG(nu_nota_redacao)::numeric, 2) AS media_redacao,
        ROUND(AVG(nu_nota_mt)::numeric, 2)      AS media_mt
    FROM tru.enem_candidatos
    WHERE q006 IS NOT NULL
    GROUP BY q006, nu_ano
    ORDER BY q006
"""
df_renda = pd.read_sql(query_renda, engine)
df_renda.to_sql("enem_renda_desempenho", engine, schema="ref",
                if_exists="replace", index=False)
print(f"ref.enem_renda_desempenho — {len(df_renda)} linhas")